# Groups

In [ ]:
# datascience
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
warnings.simplefilter(action='ignore',category=np.exceptions.VisibleDeprecationWarning)


In [ ]:
# polars
import polars as pl
from IPython.display import display
import hvplot.polars


## Prediction Example

In [ ]:
# datascience
families = Table.read_table('family_heights.csv')
families


In [ ]:
# polars
families_pl = pl.read_csv('family_heights.csv', infer_schema_length=10000)
families_pl


In [ ]:
# datascience
parent_avgs = (families.column('father') + families.column('mother'))/2


In [ ]:
# polars
parent_avgs_pl = (families_pl.get_column('father') + families_pl.get_column('mother'))/2


In [ ]:
# datascience
heights = Table().with_columns(
    'Parent Average', parent_avgs,
    'Child', families.column('child'), # At adulthood
    'M/F', families.column('child m/f')
)
heights


In [ ]:
# polars
heights_pl = pl.DataFrame([
    (parent_avgs_pl).alias('Parent Average'),
    (families_pl.get_column('child')).alias('Child'),
    (families_pl.get_column('child m/f')).alias('M/F')
])
heights_pl


In [ ]:
# datascience
heights.scatter('Parent Average', 'Child')


In [ ]:
# polars
# hvplot
heights_pl.hvplot.scatter(x='Parent Average', y='Child')


In [ ]:
# datascience
heights.scatter('Parent Average', 'Child')
plots.plot([67.5, 67.5], [50, 85], color='red', lw=2)
plots.plot([68.5, 68.5], [50, 85], color='red', lw=2);


In [ ]:
# polars
# matplotlib
plots.scatter(heights_pl.get_column('Parent Average'), heights_pl.get_column('Child'))
plots.plot([67.5, 67.5], [50, 85], color='red', lw=2)
plots.plot([68.5, 68.5], [50, 85], color='red', lw=2);


In [ ]:
# datascience
nearby = heights.where('Parent Average', are.between(67.5, 68.5))
nearby_mean = np.average(nearby.column('Child'))
nearby_mean


In [ ]:
# polars
nearby_pl = heights_pl.filter(pl.col('Parent Average').is_between(67.5, 68.5, closed='left'))
nearby_mean_pl = (nearby_pl.get_column('Child')).mean()
nearby_mean_pl


In [ ]:
# datascience
heights.scatter('Parent Average', 'Child')
plots.plot([67.5, 67.5], [50, 85], color='red', lw=2)
plots.plot([68.5, 68.5], [50, 85], color='red', lw=2)
plots.scatter(68, nearby_mean, color='red', s=50);


In [ ]:
# polars
# matplotlib
plots.scatter(heights_pl.get_column('Parent Average'), heights_pl.get_column('Child'))
plots.plot([67.5, 67.5], [50, 85], color='red', lw=2)
plots.plot([68.5, 68.5], [50, 85], color='red', lw=2)
plots.scatter(68, nearby_mean_pl, color='red', s=50);


<br/><br/><br/>

### How can we predict the height of an adult child from their parents' average height?

In [ ]:
# datascience
def predict(h):
    nearby = heights.where('Parent Average', are.between(h - 1/2, h + 1/2))
    return np.average(nearby.column('Child'))


In [ ]:
# polars
def predict_pl(h):
    nearby_pl = heights_pl.filter(pl.col('Parent Average').is_between(h - 1/2, h + 1/2, closed='left'))
    return (nearby_pl.get_column('Child')).mean()


In [ ]:
predict(68)

In [ ]:
predict(70)

In [ ]:
predict(73)

In [ ]:
# datascience
predicted_heights = heights.apply(predict, 'Parent Average')


In [ ]:
# polars
predicted_heights_pl = heights_pl.get_column('Parent Average').map_elements(predict_pl, return_dtype=pl.Float64)


In [ ]:
# datascience
heights = heights.with_column('Prediction', predicted_heights)
heights


In [ ]:
# polars
heights_pl = heights_pl.with_columns((predicted_heights_pl).alias('Prediction'))
heights_pl


In [ ]:
# datascience
heights.select('Parent Average', 'Child', 'Prediction').scatter('Parent Average')


In [ ]:
# polars
# hvplot
(lambda _df: _df.hvplot.scatter(x='Parent Average', y=[c for c in _df.columns if c != 'Parent Average']))(heights_pl.select('Parent Average', 'Child', 'Prediction'))


### Prediction Accuracy

In [ ]:
# datascience
pred_errs = heights.column('Prediction') - heights.column('Child')
heights = heights.with_column('errors',pred_errs)
heights


In [ ]:
# polars
pred_errs_pl = heights_pl.get_column('Prediction') - heights_pl.get_column('Child')
heights_pl = heights_pl.with_columns(pl.Series('errors', pred_errs_pl))
heights_pl


In [ ]:
# datascience
heights.hist('errors')


In [ ]:
# polars
# hvplot
heights_pl.hvplot.hist(y='errors', normed=True)


In [ ]:
# datascience
heights.hist('errors', group='M/F')


In [ ]:
# polars
# hvplot
heights_pl.hvplot.hist(y='errors', normed=True, by='M/F')


**DISCUSS**: What does this chart visualize?

### Predict by height and sex

Find the average of the neighbors from the same sex

In [ ]:
# datascience
def predict_smarter(h, s):
    nearby = heights.where('Parent Average', are.between(h - 1/2, h + 1/2))
    nearby_same_mf = nearby.where('M/F', s)
    return np.average(nearby_same_mf.column('Child'))


In [ ]:
# polars
def predict_smarter_pl(h, s):
    nearby_pl = heights_pl.filter(pl.col('Parent Average').is_between(h - 1/2, h + 1/2, closed='left'))
    nearby_same_mf = nearby_pl.filter((pl.col('M/F') == (s)))
    return (nearby_same_mf.get_column('Child')).mean()


In [ ]:
predict_smarter(68, 'female')

In [ ]:
predict_smarter(68, 'male')

In [ ]:
# datascience
heights.apply(predict_smarter, "Parent Average", "M/F")


In [ ]:
# polars
heights_pl.select(pl.struct(["Parent Average", "M/F"]).map_elements(lambda row: predict_smarter_pl(row["Parent Average"], row["M/F"]), return_dtype=pl.Float64)).to_series()


In [ ]:
# datascience
smarter_predicted_heights = heights.apply(predict_smarter, 'Parent Average', 'M/F')
heights = heights.with_column('Smarter Prediction', smarter_predicted_heights)
heights


In [ ]:
# polars
smarter_predicted_heights_pl = heights_pl.select(pl.struct(['Parent Average', 'M/F']).map_elements(lambda row: predict_smarter_pl(row['Parent Average'], row['M/F']), return_dtype=pl.Float64)).to_series()
heights_pl = heights_pl.with_columns(pl.Series('Smarter Prediction', smarter_predicted_heights_pl))
heights_pl


In [ ]:
# datascience
def difference(child, prediction):
    return prediction - child

smarter_pred_errs = heights.apply(difference, 'Child', 'Smarter Prediction')
heights = heights.with_column('Smarter Errors', smarter_pred_errs)


In [ ]:
# polars
smarter_pred_errs_pl = heights_pl.select(pl.struct(['Child', 'Smarter Prediction']).map_elements(lambda row: difference(row['Child'], row['Smarter Prediction']), return_dtype=pl.Float64)).to_series()
heights_pl = heights_pl.with_columns(pl.Series('Smarter Errors', smarter_pred_errs_pl))


In [ ]:
# datascience
heights.hist('Smarter Errors', group='M/F')


In [ ]:
# polars
# hvplot
heights_pl.hvplot.hist(y='Smarter Errors', normed=True, by='M/F')


## Grouping by One Column

In [ ]:
# datascience
cones = Table.read_table('cones.csv').drop('Color')
cones


In [ ]:
# polars
cones_pl = pl.read_csv('cones.csv').drop('Color')
cones_pl


In [ ]:
# datascience
cones.group('Flavor')


In [ ]:
# polars
cones_pl.group_by('Flavor').len(name='count').sort('Flavor')


In [ ]:
# datascience
cones.group('Flavor', np.average)


In [ ]:
# polars
(lambda _df: _df.group_by('Flavor').agg([getattr(pl.col(c), 'mean')().alias(f'{c} average') for c, t in _df.schema.items() if c not in ('Flavor' if isinstance('Flavor', list) else ['Flavor']) and (t.is_numeric())]).sort('Flavor'))(cones_pl)


In [ ]:
# datascience
cones.group('Flavor', np.min)


In [ ]:
# polars
(lambda _df: _df.group_by('Flavor').agg([pl.col(c).min().alias(f'{c} min') for c, t in _df.schema.items() if c not in ('Flavor' if isinstance('Flavor', list) else ['Flavor']) and (getattr(t, 'is_numeric', lambda: True)())]).sort('Flavor'))(cones_pl)


In [ ]:
# datascience
cones.group('Flavor', np.array)


In [ ]:
# polars
(lambda _df: _df.group_by('Flavor').agg([pl.col(c).alias(f'{c} array') for c, t in _df.schema.items() if c not in ('Flavor' if isinstance('Flavor', list) else ['Flavor']) and (True)]).sort('Flavor'))(cones_pl)


### Grouping By One Column: Welcome Survey

In [ ]:
# datascience
survey = Table.read_table('welcome_survey_fa26.csv')
survey.show(3)


In [ ]:
# polars
survey_pl = pl.read_csv('welcome_survey_fa26.csv')
survey_pl.head(3)


In [ ]:
# datascience
survey.num_rows


In [ ]:
# polars
survey_pl.height


In [ ]:
# datascience
survey.hist('Extroversion')


In [ ]:
# polars
# hvplot
survey_pl.hvplot.hist(y='Extroversion', normed=True)


In [ ]:
# datascience
survey.group('Extroversion', np.average)


In [ ]:
# polars
(lambda _df: _df.group_by('Extroversion').agg([getattr(pl.col(c), 'mean')().alias(f'{c} average') for c, t in _df.schema.items() if c not in ('Extroversion' if isinstance('Extroversion', list) else ['Extroversion']) and (t.is_numeric())]).sort('Extroversion'))(survey_pl)


In [ ]:
# datascience
survey.group('Extroversion', np.average).plot('Extroversion', 'Text Recipients average')


In [ ]:
# polars
# hvplot
(lambda _df: _df.group_by('Extroversion').agg([getattr(pl.col(c), 'mean')().alias(f'{c} average') for c, t in _df.schema.items() if c not in ('Extroversion' if isinstance('Extroversion', list) else ['Extroversion']) and (t.is_numeric())]).sort('Extroversion'))(survey_pl).hvplot.line(x='Extroversion', y='Text Recipients average')


## Lists

In [ ]:
[1, 5, 'hello', 5.0]

In [ ]:
# datascience
[1, 5, 'hello', 5.0, make_array(1,2,3)]


In [ ]:
# polars
[1, 5, 'hello', 5.0, pl.Series('values', [1,2,3])]


## Grouping by Two Columns

Do right-handed people tend to sleep on their left side and left-handed people sleep on their right?

In [ ]:
# datascience
survey.group('Sleep Position')


In [ ]:
# polars
survey_pl.group_by('Sleep Position').len(name='count').sort('Sleep Position')


In [ ]:
# datascience
survey.group(['Handedness', 'Sleep Position']).show()


In [ ]:
# polars
with pl.Config(tbl_rows=-1):
    display(survey_pl.group_by(['Handedness', 'Sleep Position']).len(name='count').sort(['Handedness', 'Sleep Position']))
